# UE Data Mining - Projet
Notebook à compléter dans le cadre du projet de l’UE Data Mining des Master 2 Intelligence Artificielle et Master 2 Data Science de l’année scolaire 2026-2027.

* Nom du groupe : 12
* Membres du groupe :
    * idil MAHAMOUD GUEDI
    * André da Conceição
    * Ali Chabaka 
    * Aissam Eddine Boukhelkhal
    * Lucas Duchamp
    * RATOANDROMANANA Iavosoa Miella

## Data Cleaning — Public Records

Ce notebook est dédié au nettoyage et à la préparation des **5 premiers jeux de données (par ordre alphabétique)** du dossier `data/public_records/` :
1. `institutional_memos.csv`
2. `licensed_facilities.csv`
3. `licensed_professionals.csv`
4. `officer_filings.csv`
5. `public_and_investigative_documents.csv`

### Démarche méthodologique adoptée :
* **Détection des doublons** : Identification et suppression des doublons stricts et vérification de l'unicité des identifiants métiers.
* **Traitement des valeurs manquantes** : Analyse des causes (absences structurelles pour les licences actives vs absences relationnelles) et imputation tracée ou création d'indicateurs binaires.
* **Harmonisation des chaînes de caractères** : Élimination des espaces superflus (début/fin et espaces multiples au sein des chaînes).
* **Standardisation des types et formats temporels** : Conversion des dates textuelles en objets `datetime` avec gestion des formats mixtes (`YYYY-MM-DD`, `DD/MM/YYYY`, slashes).
* **Encodage des variables catégorielles** : Création de variables binaires (One-Hot Encoding via `pd.get_dummies` et binarisation multi-label).
* **Sauvegarde** : Export des jeux de données nettoyés dans le dossier `data_cleaned/public_records/`.

In [1]:
### Imports et configuration
import pandas as pd
import numpy as np
import re
from pathlib import Path

data_dir = Path("data/public_records")
output_dir = Path("data_cleaned/public_records")
output_dir.mkdir(parents=True, exist_ok=True)

def clean_whitespace(val):
    """Supprime les espaces superflus (initiaux, finaux et consécutifs multiples)."""
    if pd.isna(val):
        return val
    s = str(val).strip()
    return re.sub(r'\s+', ' ', s)

def clean_dataframe_strings(df, exclude_cols=None):
    """Normalise les espaces pour toutes les colonnes textuelles d'un DataFrame."""
    if exclude_cols is None:
        exclude_cols = []
    df_clean = df.copy()
    for col in df_clean.select_dtypes(include='object').columns:
        if col not in exclude_cols:
            df_clean[col] = df_clean[col].apply(clean_whitespace)
    return df_clean

print("Configuration prête et fonctions utilitaires initialisées.")

Configuration prête et fonctions utilitaires initialisées.



### 1. institutional_memos

Ce jeu de données contient des notes de service internes et mémorandums d'opérations.

**Diagnostic qualité :**
* 0 doublon complet, 30 identifiants uniques.
* Espaces anormaux dans `subject_line` (ex: `'  Internal  Operations  Memorandum  57 '`).
* Valeurs manquantes relationnelles sur `related_purchase_order`, `related_shipment_number`, `related_case_number` (imputation par `'none'` et création d'indicateurs de présence `has_related_*`).
* Date à convertir en `datetime` et encodage disjonctif (`pd.get_dummies`) de `document_type` et `visibility_level`.

In [2]:
### institutional_memos — Exploration
df_memos = pd.read_csv(data_dir / "institutional_memos.csv")
print("Dimensions :", df_memos.shape)
print("Doublons :", df_memos.duplicated().sum())
print("\nValeurs manquantes :\n", df_memos.isna().sum())

# Détection des espaces parasites dans subject_line
dirty_sub = df_memos[df_memos["subject_line"].str.contains(r"^\s+|\s+$|\s{2,}", regex=True)]
print("\nLignes avec espaces anormaux dans subject_line :")
for idx, r in dirty_sub.iterrows():
    print(f"Ligne {idx} : {repr(r['subject_line'])}")

Dimensions : (30, 12)
Doublons : 0

Valeurs manquantes :
 document_id                 0
document_date               0
document_type               0
author_name                 0
author_organization         0
recipient_text              0
subject_line                0
text                        0
related_purchase_order     25
related_shipment_number    22
related_case_number        20
visibility_level            0
dtype: int64

Lignes avec espaces anormaux dans subject_line :
Ligne 4 : '  Internal  Operations  Memorandum  57 '



In [3]:
### institutional_memos — Nettoyage
# 1. Normalisation des espaces
df_memos = clean_dataframe_strings(df_memos, exclude_cols=["text"])
df_memos["text"] = df_memos["text"].str.strip()

# 2. Conversion des dates
df_memos["document_date"] = pd.to_datetime(df_memos["document_date"], errors="coerce")

# 3. Drapeaux booléens et imputation des clés relationnelles
df_memos["has_related_purchase_order"] = df_memos["related_purchase_order"].notna().astype(int)
df_memos["has_related_shipment_number"] = df_memos["related_shipment_number"].notna().astype(int)
df_memos["has_related_case_number"] = df_memos["related_case_number"].notna().astype(int)

for col in ["related_purchase_order", "related_shipment_number", "related_case_number"]:
    df_memos[col] = df_memos[col].fillna("none")

# 4. Encodage catégoriel
dummies_type = pd.get_dummies(df_memos["document_type"], prefix="doc_type", dtype=int)
dummies_vis = pd.get_dummies(df_memos["visibility_level"], prefix="visibility", dtype=int)
df_memos_clean = pd.concat([df_memos, dummies_type, dummies_vis], axis=1)

print("Dimensions après nettoyage :", df_memos_clean.shape)
print("Valeurs manquantes restantes :", df_memos_clean.isna().sum().sum())
print("Valeur corrigée subject_line :", repr(df_memos_clean.loc[4, "subject_line"]))
df_memos_clean.head(2)

Dimensions après nettoyage : (30, 24)
Valeurs manquantes restantes : 0
Valeur corrigée subject_line : 'Internal Operations Memorandum 57'



### 2. licensed_facilities

Ce jeu de données recense les établissements de santé, cliniques et laboratoires agréés.

**Diagnostic qualité :**
* 3 établissements enregistrés.
* Colonne `license_end` vide à 100% (licences permanentes ou actives) -> suppression de la colonne.
* Date de début de licence `license_start` à convertir en `datetime`.
* La colonne `declared_services` contient plusieurs services séparés par des points-virgules -> binarisation multi-label.
* Encodage disjonctif de `facility_category` et `region`.

In [4]:
### licensed_facilities — Exploration
df_fac = pd.read_csv(data_dir / "licensed_facilities.csv")
print("Dimensions :", df_fac.shape)
print("\nInformations :")
df_fac.info()
print("\nAperçu :")
df_fac[["facility_license_number", "facility_name", "license_start", "license_end", "facility_status", "declared_services"]]

Dimensions : (3, 12)

Informations :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 12 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   facility_license_number              3 non-null      object 
 1   facility_name                        3 non-null      object 
 2   facility_category                    3 non-null      object 
 3   operating_organization_registration  3 non-null      object 
 4   street_address                       3 non-null      object 
 5   region                               3 non-null      object 
 6   latitude                             3 non-null      float64
 7   longitude                            3 non-null      float64
 8   license_start                        3 non-null      object 
 9   license_end                          0 non-null      float64
 10  facility_status                      3 non-null      object 
 11 

In [5]:
### licensed_facilities — Nettoyage
# 1. Suppression de la colonne entièrement vide
df_fac = df_fac.drop(columns=["license_end"])

# 2. Normalisation des espaces
df_fac = clean_dataframe_strings(df_fac)

# 3. Conversion de la date
df_fac["license_start"] = pd.to_datetime(df_fac["license_start"], errors="coerce")

# 4. Extraction multi-label des services déclarés
all_services = sorted(list(set(
    s.strip()
    for sublist in df_fac["declared_services"].str.split(";")
    for s in sublist
)))

for s in all_services:
    col_name = "service_" + re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
    df_fac[col_name] = df_fac["declared_services"].apply(
        lambda x: int(s in [item.strip() for item in x.split(";")])
    )

# 5. Encodage catégoriel
dummies_cat = pd.get_dummies(df_fac["facility_category"], prefix="facility_cat", dtype=int)
dummies_reg = pd.get_dummies(df_fac["region"], prefix="facility_region", dtype=int)
df_fac_clean = pd.concat([df_fac, dummies_cat, dummies_reg], axis=1)

print("Dimensions après nettoyage :", df_fac_clean.shape)
print("Valeurs manquantes restantes :", df_fac_clean.isna().sum().sum())
print("Colonnes de services créées :", [c for c in df_fac_clean.columns if c.startswith("service_")])
df_fac_clean.head(3)

Dimensions après nettoyage : (3, 24)
Valeurs manquantes restantes : 0
Colonnes de services créées : ['service_cardiac_assessment', 'service_clinical_research', 'service_environmental_health_research', 'service_executive_health', 'service_non_invasive_human_assessment', 'service_outpatient_procedures', 'service_preventive_medicine', 'service_specialist_follow_up']



### 3. licensed_professionals

Ce jeu de données liste les praticiens, chercheurs et professionnels accrédités.

**Diagnostic qualité :**
* 53 professionnels uniques, aucun doublon.
* Espace parasite dans `specialty` à la ligne 36 (`'  environmental  assessment '`) créant un doublon de modalité -> normalisation des espaces.
* Format de date non standard à la ligne 26 (`01/07/1994`, format européen DD/MM/YYYY) au lieu du format ISO -> conversion avec `format='mixed'` et `dayfirst=True`.
* `license_expiry_date` vide à 100% -> suppression de la colonne.
* 26 valeurs manquantes dans `primary_facility_license` correspondant aux professionnels non hospitaliers (administratifs, juristes) -> imputation par `'unaffiliated'`.
* Encodage disjonctif de `profession`, `registered_region` et `disciplinary_status`.

In [6]:
### licensed_professionals — Exploration
df_prof = pd.read_csv(data_dir / "licensed_professionals.csv")
print("Dimensions :", df_prof.shape)
print("\nValeurs manquantes :\n", df_prof.isna().sum())

# Mise en évidence de l'anomalie d'espace dans specialty
print("\nModalités de specialty avant nettoyage :")
print(df_prof["specialty"].value_counts())

# Format de date européen non standard ligne 26
print("\nLigne 26 - Date non standard :", repr(df_prof.loc[26, "license_issue_date"]))

Dimensions : (53, 10)

Valeurs manquantes :
 license_number               0
professional_name            0
profession                   0
specialty                    0
license_issue_date           0
license_expiry_date         53
license_status               0
primary_facility_license    26
registered_region            0
disciplinary_status          0
dtype: int64

Modalités de specialty avant nettoyage :
specialty
programme administration        21
public policy research           6
regulatory compliance            5
environmental assessment         5
wildlife monitoring              3
clinical research                2
cardiology                       2
preventive medicine              1
occupational medicine            1
ecological analytics             1
cardiac imaging                  1
medical audit                    1
pollinator ecology               1
general medicine                 1
  environmental  assessment      1
clinical physiology              1
Name: count, dtype: 

In [7]:
### licensed_professionals — Nettoyage
# 1. Normalisation des espaces (résout l'anomalie dans specialty)
df_prof = clean_dataframe_strings(df_prof)

# 2. Conversion de license_issue_date avec format mixte et dayfirst=True
df_prof["license_issue_date"] = pd.to_datetime(
    df_prof["license_issue_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# 3. Suppression de la colonne 100% vide license_expiry_date
df_prof = df_prof.drop(columns=["license_expiry_date"])

# 4. Imputation des professionnels non rattachés à un établissement
df_prof["primary_facility_license"] = df_prof["primary_facility_license"].fillna("unaffiliated")

# 5. Encodage catégoriel
dummies_prof = pd.get_dummies(df_prof["profession"], prefix="profession", dtype=int)
dummies_reg = pd.get_dummies(df_prof["registered_region"], prefix="region", dtype=int)
dummies_disc = pd.get_dummies(df_prof["disciplinary_status"], prefix="discipline", dtype=int)
df_prof_clean = pd.concat([df_prof, dummies_prof, dummies_reg, dummies_disc], axis=1)

print("Dimensions après nettoyage :", df_prof_clean.shape)
print("Valeurs manquantes restantes :", df_prof_clean.isna().sum().sum())
print("Date ligne 26 convertie :", df_prof_clean.loc[26, "license_issue_date"])
print("Nombre de 'environmental assessment' unifiés :", (df_prof_clean["specialty"] == "environmental assessment").sum())
df_prof_clean.head(2)

Dimensions après nettoyage : (53, 21)
Valeurs manquantes restantes : 0
Date ligne 26 convertie : 1994-07-01 00:00:00
Nombre de 'environmental assessment' unifiés : 6



### 4. officer_filings

Ce jeu de données recense les mandats d'administrateurs et dirigeants enregistrés.

**Diagnostic qualité :**
* 1 doublon exact (lignes 52 et 135, mandat `FLG-000053` de Nadia Aren) -> suppression du doublon.
* Espaces anormaux dans `full_name` (ligne 98 : `'  Pavel  Waren '`) -> normalisation.
* Format de date avec des slashes dans `appointment_date` (ligne 59 : `'1986/01/01'`) -> conversion robuste.
* `cessation_date` contient 77 valeurs manquantes correspondant exactement aux 77 mandats actifs (`filing_status == 'active'`).
* `birth_year_or_age_band` mélange des années exactes et des tranches d'âges (ex. `1980–1989`) -> parsing en deux colonnes numériques `birth_year_min` et `birth_year_max`.
* Encodage disjonctif de `nationality_or_region` et `filing_status`.

In [8]:
### officer_filings — Exploration
df_filings = pd.read_csv(data_dir / "officer_filings.csv")
print("Dimensions :", df_filings.shape)
print("Doublons stricts :", df_filings.duplicated().sum())

# Affichage du doublon
dup_rows = df_filings[df_filings.duplicated(keep=False)]
print("\nLignes dupliquées détectées :")
print(dup_rows[["filing_id", "full_name", "role_title", "appointment_date"]])

# Détection des anomalies d'espace et de slash
dirty_n = df_filings[df_filings["full_name"].str.contains(r"^\s+|\s+$|\s{2,}", regex=True)]
print("\nNom avec espaces superflus :", dirty_n["full_name"].tolist())

slash_d = df_filings[df_filings["appointment_date"].str.contains("/", na=False)]
print("Date avec slashes :", slash_d["appointment_date"].tolist())

Dimensions : (136, 12)
Doublons stricts : 1

Lignes dupliquées détectées :
      filing_id   full_name          role_title appointment_date
52   FLG-000053  Nadia Aren  Operations manager       2010-01-01
135  FLG-000053  Nadia Aren  Operations manager       2010-01-01

Nom avec espaces superflus : ['  Pavel  Waren ']
Date avec slashes : ['1986/01/01']



In [9]:
### officer_filings — Nettoyage
# 1. Élimination du doublon strict
df_filings = df_filings.drop_duplicates().reset_index(drop=True)

# 2. Normalisation des espaces
df_filings = clean_dataframe_strings(df_filings)

# 3. Conversion robuste des dates
for date_col in ["appointment_date", "filing_date", "cessation_date"]:
    df_filings[date_col] = pd.to_datetime(df_filings[date_col], format="mixed", errors="coerce")

# 4. Parsing de birth_year_or_age_band en bornes min/max
def parse_birth_band(val):
    if pd.isna(val):
        return None, None
    s = str(val).strip().replace('\u2013', '-').replace('—', '-')
    if '-' in s:
        parts = s.split('-')
        return int(parts[0]), int(parts[1])
    else:
        y = int(s)
        return y, y

df_filings["birth_year_min"] = df_filings["birth_year_or_age_band"].apply(lambda x: parse_birth_band(x)[0])
df_filings["birth_year_max"] = df_filings["birth_year_or_age_band"].apply(lambda x: parse_birth_band(x)[1])

# 5. Encodage catégoriel
dummies_nat = pd.get_dummies(df_filings["nationality_or_region"], prefix="nationality", dtype=int)
dummies_status = pd.get_dummies(df_filings["filing_status"], prefix="status", dtype=int)
df_filings_clean = pd.concat([df_filings, dummies_nat, dummies_status], axis=1)

print("Dimensions après nettoyage :", df_filings_clean.shape)
print("Doublons restants :", df_filings_clean.duplicated().sum())
print("Valeurs manquantes dans cessation_date :", df_filings_clean["cessation_date"].isna().sum())
print("Mandats actifs correspondants :", (df_filings_clean["filing_status"] == "active").sum())
df_filings_clean.head(2)

Dimensions après nettoyage : (135, 21)
Doublons restants : 0
Valeurs manquantes dans cessation_date : 77
Mandats actifs correspondants : 77



### 5. public_and_investigative_documents

Ce jeu de données rassemble des rapports d'investigation, lettres ouvertes et synthèses de recherche d'intérêt public.

**Diagnostic qualité :**
* 40 documents, 0 doublon.
* 1 valeur manquante dans `author_name` (ligne 24, `PUB-510056`, lettre collective de *Threshold Earth Observatory*) -> imputation par `'unknown'`.
* Normalisation des espaces dans les métadonnées et `strip()` sur le corps du texte (`text`).
* Conversion de `document_date` en `datetime`.
* Encodage disjonctif de `collection` et `public_or_restricted`.

In [10]:
### public_and_investigative_documents — Exploration
df_docs = pd.read_csv(data_dir / "public_and_investigative_documents.csv")
print("Dimensions :", df_docs.shape)
print("Valeurs manquantes :\n", df_docs.isna().sum())

# Examen du document sans auteur physique individuel (index 24)
missing_auth = df_docs[df_docs["author_name"].isna()]
print("\nDocument avec author_name manquant :")
print(missing_auth[["document_id", "document_date", "author_organization", "title", "public_or_restricted"]])

Dimensions : (40, 11)
Valeurs manquantes :
 document_id             0
document_date           0
collection              0
document_type           0
author_name             1
author_organization     0
named_location          0
title                   0
text                    0
public_or_restricted    0
source_reference        0
dtype: int64

Document avec author_name manquant :
   document_id  ... public_or_restricted
24  PUB-510056  ...               public

[1 rows x 5 columns]



In [11]:
### public_and_investigative_documents — Nettoyage
# 1. Imputation de l'auteur manquant
df_docs["author_name"] = df_docs["author_name"].fillna("unknown")

# 2. Normalisation des espaces
df_docs = clean_dataframe_strings(df_docs, exclude_cols=["text"])
df_docs["text"] = df_docs["text"].str.strip()

# 3. Conversion de la date
df_docs["document_date"] = pd.to_datetime(df_docs["document_date"], errors="coerce")

# 4. Encodage catégoriel
dummies_coll = pd.get_dummies(df_docs["collection"], prefix="collection", dtype=int)
dummies_vis = pd.get_dummies(df_docs["public_or_restricted"], prefix="access", dtype=int)
df_docs_clean = pd.concat([df_docs, dummies_coll, dummies_vis], axis=1)

print("Dimensions après nettoyage :", df_docs_clean.shape)
print("Valeurs manquantes restantes :", df_docs_clean.isna().sum().sum())
print("Auteur après imputation (index 24) :", df_docs_clean.loc[24, "author_name"])
df_docs_clean.head(2)

Dimensions après nettoyage : (40, 19)
Valeurs manquantes restantes : 0
Auteur après imputation (index 24) : unknown



### 6. Sauvegarde et synthèse des données nettoyées

Exportation des jeux de données nettoyés dans `data_cleaned/public_records/` et tableau récapitulatif des opérations effectuées.

In [12]:
# Export vers data_cleaned/public_records/
datasets_to_save = {
    "institutional_memos_clean.csv": df_memos_clean,
    "licensed_facilities_clean.csv": df_fac_clean,
    "licensed_professionals_clean.csv": df_prof_clean,
    "officer_filings_clean.csv": df_filings_clean,
    "public_and_investigative_documents_clean.csv": df_docs_clean,
}

for file_name, df_clean in datasets_to_save.items():
    dest_path = output_dir / file_name
    df_clean.to_csv(dest_path, index=False)
    print(f"Exporté avec succès : {dest_path} ({df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes)")

# Synthèse des opérations
summary_data = [
    {
        "Dataset": "institutional_memos",
        "Lignes initiales": len(df_memos),
        "Lignes finales": len(df_memos_clean),
        "Colonnes finales": df_memos_clean.shape[1],
        "Doublons supprimés": 0,
        "Actions": "Nettoyage espaces (subject_line), conversion dates, drapeaux booléens pour clés, encodage dummies"
    },
    {
        "Dataset": "licensed_facilities",
        "Lignes initiales": len(df_fac),
        "Lignes finales": len(df_fac_clean),
        "Colonnes finales": df_fac_clean.shape[1],
        "Doublons supprimés": 0,
        "Actions": "Suppression colonne vide license_end, conversion dates, binarisation multi-label des services, encodage dummies"
    },
    {
        "Dataset": "licensed_professionals",
        "Lignes initiales": len(df_prof),
        "Lignes finales": len(df_prof_clean),
        "Colonnes finales": df_prof_clean.shape[1],
        "Doublons supprimés": 0,
        "Actions": "Harmonisation specialty ('environmental assessment'), parsing date mixte DD/MM/YYYY, suppression license_expiry_date, imputation primary_facility_license, encodage dummies"
    },
    {
        "Dataset": "officer_filings",
        "Lignes initiales": 136,
        "Lignes finales": len(df_filings_clean),
        "Colonnes finales": df_filings_clean.shape[1],
        "Doublons supprimés": 1,
        "Actions": "Élimination doublon exact FLG-000053, correction espaces Pavel Waren, normalisation dates avec slashes, parsing tranches d'âge en min/max, encodage dummies"
    },
    {
        "Dataset": "public_and_investigative_documents",
        "Lignes initiales": len(df_docs),
        "Lignes finales": len(df_docs_clean),
        "Colonnes finales": df_docs_clean.shape[1],
        "Doublons supprimés": 0,
        "Actions": "Imputation author_name manquant ('unknown'), normalisation des textes, conversion dates, encodage dummies access/collection"
    },
]

df_summary = pd.DataFrame(summary_data)
print("\n--- Synthèse du Data Cleaning (Public Records) ---")
print(df_summary.to_string(index=False))

Exporté avec succès : data_cleaned/public_records/institutional_memos_clean.csv (30 lignes, 24 colonnes)
Exporté avec succès : data_cleaned/public_records/licensed_facilities_clean.csv (3 lignes, 24 colonnes)
Exporté avec succès : data_cleaned/public_records/licensed_professionals_clean.csv (53 lignes, 21 colonnes)
Exporté avec succès : data_cleaned/public_records/officer_filings_clean.csv (135 lignes, 21 colonnes)
Exporté avec succès : data_cleaned/public_records/public_and_investigative_documents_clean.csv (40 lignes, 19 colonnes)

--- Synthèse du Data Cleaning (Public Records) ---
                           Dataset  Lignes initiales  Lignes finales  Colonnes finales  Doublons supprimés                                                                                                                                                                     Actions
               institutional_memos                30              30                24                   0                        